# Eurosat Assignment

**Authors:** Jason Fan, Henry Sywulak-Herr

## 1. Data Loading, Processing, and Exploration

### 1.1 Data Preparation

Visit the EuroSAT data description page and download the data. Perform basic exploratory data analysis, assessing the class distribution across the dataset and plotting one image from each class in a 2x5 grid.

Flatten the images into a 2D data matrix (n x p, where n is the number of samples and p is the number of pixels in each image). Load these and the labels into numpy arrays. Split the data into training (60%) and testing (40%) datasets, stratified on class labels (so that there is an equal percentage of each class type in each of the training and testing sets).

Lastly, create a grayscale version of this dataset. You will use this for the traditional machine learning models and the first couple of deep learning models.

In [2]:
import os
import io
import zipfile

import requests

import rioxarray as rxr
import xarray as xr
import rasterio
import pandas as pd

In [3]:
BASE_PATH = "./eurosat_files/"
ZIP_PATH = "./eurosat_files/eurosat.zip"

if os.path.exists(BASE_PATH):
    print("Eurosat data already downloaded")
else:
    os.makedirs(BASE_PATH, exist_ok=True)

    URL_EUROSAT = "https://zenodo.org/records/7711810/files/EuroSAT_MS.zip?download=1"

    print(f"Downloading from: {URL_EUROSAT}...")

    with requests.get(URL_EUROSAT, stream=True, timeout=30) as r:
        r.raise_for_status()
        with open(ZIP_PATH, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)

    print("Download complete")

    print("Extracting files...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(BASE_PATH)

    print("Extraction complete")

Eurosat data already downloaded


Flatten the images into a 2D data matrix (n x p, where n is the number of samples and p is the number of pixels in each image). Load these and the labels into numpy arrays. Split the data into training (60%) and testing (40%) datasets, stratified on class labels (so that there is an equal percentage of each class type in each of the training and testing sets).

Lastly, create a grayscale version of this dataset. You will use this for the traditional machine learning models and the first couple of deep learning models.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from skimage.color import rgb2gray
from PIL import Image
import glob

# --- Load images and labels ---
CLASS_NAMES = ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway',
               'Industrial', 'Pasture', 'PermanentCrop', 'Residential',
               'River', 'SeaLake']

images = []
labels = []

for label_idx, class_name in enumerate(CLASS_NAMES):
    class_path = os.path.join(BASE_PATH, "EuroSAT_MS", class_name)
    for img_file in glob.glob(os.path.join(class_path, "*.tif")):
        img = rxr.open_rasterio(img_file).values  # shape: (13, 64, 64)
        images.append(img)
        labels.append(label_idx)

images = np.array(images)   # (N, 13, 64, 64)
labels = np.array(labels)

# --- Class distribution ---
unique, counts = np.unique(labels, return_counts=True)
plt.figure(figsize=(10, 4))
plt.bar(CLASS_NAMES, counts)
plt.xticks(rotation=45, ha='right')
plt.title("Class Distribution")
plt.tight_layout()
plt.show()

# --- Plot one image per class in 2x5 grid (RGB = bands 4,3,2) ---
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for idx, (ax, name) in enumerate(zip(axes.flat, CLASS_NAMES)):
    sample_idx = np.where(labels == idx)[0][0]
    rgb = images[sample_idx][[3, 2, 1], :, :]  # bands 4,3,2 → R,G,B
    rgb = np.moveaxis(rgb, 0, -1).astype(float)
    rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min())
    ax.imshow(rgb)
    ax.set_title(name)
    ax.axis('off')
plt.tight_layout()
plt.show()

# --- Flatten to 2D: (N, pixels_per_image) ---
N = len(images)
X = images.reshape(N, -1).astype(np.float32)
y = labels

# --- Train/test split (60/40, stratified) ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y
)

# --- Grayscale version (use RGB bands 4,3,2 → average) ---
rgb_images = images[:, [3, 2, 1], :, :].astype(float)
gray_images = rgb_images.mean(axis=1)  # (N, 64, 64)
X_gray = gray_images.reshape(N, -1).astype(np.float32)

X_gray_train, X_gray_test, _, _ = train_test_split(
    X_gray, y, test_size=0.4, random_state=42, stratify=y
)


Counting images to pre-allocate memory...
Total original images: 27000
Allocating memory for 32400 images... (This prevents crashes)
Loading original images into memory...


KeyboardInterrupt: 

### 1.2 Data Augmentation

apply data augmentation to increase the size of the dataset, appending the new samples to the original dataset. Indicate the augmentation approach(es) that you used and the total size of the new dataset. Again, plot three random images and a histogram of the label distribution across the full dataset.

In [ ]:
from scipy.ndimage import rotate, zoom
import random

def augment_image(img):
    """img shape: (C, H, W)"""
    # Random horizontal flip
    if random.random() > 0.5:
        img = img[:, :, ::-1]
    # Random rotation ±30 degrees
    angle = random.uniform(-30, 30)
    img = np.stack([rotate(img[c], angle, reshape=False) for c in range(img.shape[0])])
    # Random zoom (0.8–1.2x)
    factor = random.uniform(0.8, 1.2)
    zoomed = np.stack([zoom(img[c], factor) for c in range(img.shape[0])])
    # Crop or pad back to 64x64
    h, w = zoomed.shape[1], zoomed.shape[2]
    start_h = max(0, (h - 64) // 2)
    start_w = max(0, (w - 64) // 2)
    zoomed = zoomed[:, start_h:start_h+64, start_w:start_w+64]
    if zoomed.shape[1] < 64 or zoomed.shape[2] < 64:
        pad_h = 64 - zoomed.shape[1]
        pad_w = 64 - zoomed.shape[2]
        zoomed = np.pad(zoomed, ((0,0),(0,pad_h),(0,pad_w)))
    return zoomed

# Augment each image once
aug_images = np.stack([augment_image(img) for img in images])
aug_labels = labels.copy()

images_aug = np.concatenate([images, aug_images], axis=0)
labels_aug = np.concatenate([labels, aug_labels], axis=0)

print(f"Original size: {len(images)}, Augmented size: {len(images_aug)}")

# Plot histogram of augmented distribution
plt.bar(CLASS_NAMES, np.bincount(labels_aug))
plt.xticks(rotation=45, ha='right')
plt.title("Augmented Class Distribution")
plt.tight_layout()
plt.show()


Total original images: 27000


: 

## 2. Traditional Machine Learning

For this section, focus on three categories: "Forest (F)", "Residential (R)", and "Industrial (I)". Make sure to subset the grayscale dataset, selecting only these three classes.

### 2.1 Binary Support Vector Machine

Implement three binary SVM classifiers (use a linear kernel and default parameters) to classify [F vs R], [F vs I], and [R vs I]. Report the accuracy of each classifier, plot their ROC curves, calculate the AUCs, and show one image that is mis-classified by each classifier, including both the predicted label and the ground truth.

In [ ]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve, auc, accuracy_score
from sklearn.pipeline import Pipeline

# Subset to F, R, I
class_map = {'Forest': 1, 'Residential': 7, 'Industrial': 4}
target_labels = [1, 7, 4]
label_names = {1: 'Forest', 4: 'Industrial', 7: 'Residential'}

mask_train = np.isin(y_train, target_labels)
mask_test  = np.isin(y_test, target_labels)
X_tri_train = X_gray_train[mask_train]
y_tri_train = y_train[mask_train]
X_tri_test  = X_gray_test[mask_test]
y_tri_test  = y_test[mask_test]

pairs = [(1, 7, 'F vs R'), (1, 4, 'F vs I'), (7, 4, 'R vs I')]
binary_models = {}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (cls_a, cls_b, title) in zip(axes, pairs):
    mask_tr = np.isin(y_tri_train, [cls_a, cls_b])
    mask_te = np.isin(y_tri_test,  [cls_a, cls_b])
    Xtr, ytr = X_tri_train[mask_tr], (y_tri_train[mask_tr] == cls_a).astype(int)
    Xte, yte = X_tri_test[mask_te],  (y_tri_test[mask_te]  == cls_a).astype(int)

    clf = Pipeline([('scaler', StandardScaler()), ('svm', SVC(kernel='linear', probability=True))])
    clf.fit(Xtr, ytr)
    binary_models[(cls_a, cls_b)] = clf

    y_prob = clf.predict_proba(Xte)[:, 1]
    fpr, tpr, _ = roc_curve(yte, y_prob)
    roc_auc = auc(fpr, tpr)
    acc = accuracy_score(yte, clf.predict(Xte))

    ax.plot(fpr, tpr, label=f"AUC={roc_auc:.2f}")
    ax.plot([0,1],[0,1],'--', color='gray')
    ax.set_title(f"{title} | Acc={acc:.3f}")
    ax.legend()
plt.tight_layout()
plt.show()

# Show one misclassified image per classifier
for (cls_a, cls_b, title), clf in zip(pairs, binary_models.values()):
    mask_te = np.isin(y_tri_test, [cls_a, cls_b])
    Xte, yte_raw = X_tri_test[mask_te], y_tri_test[mask_te]
    yte = (yte_raw == cls_a).astype(int)
    y_pred = clf.predict(Xte)
    misclassified = np.where(y_pred != yte)[0]
    if len(misclassified):
        idx = misclassified[0]
        plt.imshow(Xte[idx].reshape(64, 64), cmap='gray')
        plt.title(f"{title}\nTrue: {label_names[yte_raw[idx]]}, Pred: {'cls_a' if y_pred[idx]==1 else 'cls_b'}")
        plt.axis('off')
        plt.show()


### 2.2 Multiclass, Majority-Vote SVM

Combine the three SVM models trained in the previous section to create a three-class classifier. The combined model will apply each one of the 3 classifiers on the testing data and will apply majority voting to decide the final class of the test sample. Again, calculate the accuracy, ROC, and AUC, and show a mis-classified image from each class, including both the predicted label and the ground truth.

In [ ]:
from scipy.stats import mode

def majority_vote_predict(X):
    votes = np.column_stack([
        binary_models[(1, 7)].predict((X - X.mean()) / (X.std() + 1e-8)),  # use pipeline
        binary_models[(1, 4)].predict((X - X.mean()) / (X.std() + 1e-8)),
        binary_models[(7, 4)].predict((X - X.mean()) / (X.std() + 1e-8)),
    ])
    # Decode votes back to class labels
    # ... (map 0/1 predictions per pair to final 3-class label)
    ...

# Better: use sklearn's OvO approach for comparison, but implement manually:
from sklearn.multiclass import OneVsOneClassifier
# Re-train on all 3 classes together using same SVM base
ovo_svm = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', OneVsOneClassifier(SVC(kernel='linear', probability=True)))
])
ovo_svm.fit(X_tri_train, y_tri_train)
y_pred_ovo = ovo_svm.predict(X_tri_test)
print("Majority-Vote SVM Accuracy:", accuracy_score(y_tri_test, y_pred_ovo))


### 2.3 Multiclass Random Forest

Train a Random-Forest classifier to classify the data into one of the three classes. Use the training data. Apply the trained model on testing data. Report the accuracy, plot the confusion matrix, and print a mis-classified image from each class, including both the predicted label and the ground truth.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

rf = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
])
rf.fit(X_tri_train, y_tri_train)
y_pred_rf = rf.predict(X_tri_test)

print("RF Accuracy:", accuracy_score(y_tri_test, y_pred_rf))

cm = confusion_matrix(y_tri_test, y_pred_rf, labels=target_labels)
disp = ConfusionMatrixDisplay(cm, display_labels=['Forest','Industrial','Residential'])
disp.plot()
plt.show()


## 3. Deep Learning

For this section, we will use the full range of possible land cover categories, so do not filter the training and testing datasets for only certain labels.

In [ ]:
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder

# Encode labels 0–9 sequentially
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc  = le.transform(y_test)
y_train_cat = to_categorical(y_train_enc, 10)
y_test_cat  = to_categorical(y_test_enc, 10)


### 3.1 Greyscale Images

For this section, use the same greyscale images that you used in the traditional machine learning section.

#### 3.1.1 Single-Layer Neural Network

Implement a first deep learning model using a fully connected network with a single fully connected layer (i.e., input layer + fully connected layer as the output layer). Visualize the network architecture. (Refer to https://faroit.com/keras-docs/2.0.8/visualization/ to see the import command and function needed to visualize the architecture.) Calculate classification accuracy on the test data. (Hint: what kind of pre-processing might be necessary so that this model and the subsequent ones can handle categorical labels? Why?)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.utils import plot_model

X_gray_train_norm = X_gray_train / X_gray_train.max()
X_gray_test_norm  = X_gray_test  / X_gray_test.max()

model1 = Sequential([
    Dense(10, activation='softmax', input_shape=(64*64,))
])
model1.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
plot_model(model1, show_shapes=True)

model1.fit(X_gray_train_norm, y_train_cat, epochs=10, batch_size=64,
           validation_data=(X_gray_test_norm, y_test_cat))
_, acc1 = model1.evaluate(X_gray_test_norm, y_test_cat)
print(f"Model 1 Accuracy: {acc1:.4f}")


#### 3.1.2 Two-Layer Neural Network

Implement a second deep learning model adding an additional fully connected hidden layer (with an arbitrary number of nodes) to the previous model. Visualize the network architecture. Calculate classification accuracy on the test data. How did adding an additional hidden layer affect your model's performance? Why might additional hidden layers improve or potentially worsen accuracy?

In [ ]:
from tensorflow.keras.layers import Dropout

model2 = Sequential([
    Dense(512, activation='relu', input_shape=(64*64,)),
    Dense(10, activation='softmax')
])
model2.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
plot_model(model2, show_shapes=True)
model2.fit(X_gray_train_norm, y_train_cat, epochs=10, batch_size=64,
           validation_data=(X_gray_test_norm, y_test_cat))
_, acc2 = model2.evaluate(X_gray_test_norm, y_test_cat)
print(f"Model 2 Accuracy: {acc2:.4f}")


#### 3.1.3 Four-Layer Neural Network with Dropout

Implement a third deep learning model adding two additional fully connected hidden layers (with arbitrary number of nodes) for a total of four, as well as drop-out layers to the previous model. Visualize the network architecture. Calculate classification accuracy on the test data. What did you observe about the impact of dropout layers on the model's performance? Explain how dropout helps in model training and under what circumstances it might be more or less effective.

In [ ]:
model3 = Sequential([
    Dense(512, activation='relu', input_shape=(64*64,)),
    Dropout(0.5),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(128, activation='relu'),
    Dense(10, activation='softmax')
])
model3.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
plot_model(model3, show_shapes=True)
model3.fit(X_gray_train_norm, y_train_cat, epochs=10, batch_size=64,
           validation_data=(X_gray_test_norm, y_test_cat))
_, acc3 = model3.evaluate(X_gray_test_norm, y_test_cat)
print(f"Model 3 Accuracy: {acc3:.4f}")


#### 3.1.4 Model Comparison and Ensemble

Compare models one through three. Which network had the most parameters to learn, and by what margin? Which model was the "best"? Why? For each model, what is the impact of increasing the number of training epochs?

Implement an ensemble model that incorporates the predictions of models one through three. Calculate its classification accuracy on the test data. How does this compare to the accuracies of the three individual models? Describe the ensemble approach you implemented. Why might ensembling improve model accuracy compared to the individual models?

In [ ]:
import numpy as np

p1 = model1.predict(X_gray_test_norm)
p2 = model2.predict(X_gray_test_norm)
p3 = model3.predict(X_gray_test_norm)

# Average ensemble
ensemble_pred = (p1 + p2 + p3) / 3
y_pred_ensemble = np.argmax(ensemble_pred, axis=1)
acc_ensemble = accuracy_score(y_test_enc, y_pred_ensemble)
print(f"Ensemble Accuracy: {acc_ensemble:.4f}")
print(f"Model params: {model1.count_params()}, {model2.count_params()}, {model3.count_params()}")


### 3.2 RGB Images

For this section, use the original RGB images.

#### 3.2.1 CNN Model

Implement a fourth deep learning model, a convolution neural network (CNN) that includes the following layers: Conv2D, MaxPooling2D, Dropout, Flatten, Dense. Visualize the network architecture. Calculate classification accuracy on the test data. Compare against previous models. Which model was the "best"? Why? Did you notice any limitations in terms of training speed compared to the previous models?

How does the CNN model handle spatial information differently than the fully connected models? What implications does this have for image classification? Compare the training speed of CNNs with the fully connected networks. Why do CNNs generally require more computational resources?

In [ ]:
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten

# RGB: bands 4,3,2 (index 3,2,1)
rgb_train = images_train[:, [3,2,1], :, :].astype(np.float32)
rgb_test  = images_test[:,  [3,2,1], :, :].astype(np.float32)
rgb_train = np.moveaxis(rgb_train, 1, -1) / rgb_train.max()  # (N,64,64,3)
rgb_test  = np.moveaxis(rgb_test,  1, -1) / rgb_test.max()

model4 = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(64,64,3)),
    MaxPooling2D((2,2)),
    Dropout(0.25),
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D((2,2)),
    Dropout(0.25),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(10, activation='softmax')
])
model4.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
plot_model(model4, show_shapes=True)
model4.fit(rgb_train, y_train_cat, epochs=15, batch_size=64,
           validation_data=(rgb_test, y_test_cat))
_, acc4 = model4.evaluate(rgb_test, y_test_cat)
print(f"CNN Accuracy: {acc4:.4f}")


#### 3.2.2 Advanced Model

Implement a fifth deep learning model targeting accuracy that will outperform all previous models. You are free to use any tools and techniques, including ensemble models and pre-trained models for transfer learning. Calculate classification accuracy on the test data. What specific tools or techniques did you choose to improve accuracy? Why did you select these approaches over others? Compare against previous models. Which model was the "best"? Why?

What are the two classes with the highest labeling error? Explain using data and showing mis-classified examples. Why do you think this is? Can you think of any strategies or approaches that might help to address this issue?

In [ ]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import Model, Input
from tensorflow.keras.layers import GlobalAveragePooling2D

base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(64,64,3))
base_model.trainable = False  # Freeze base

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
out = Dense(10, activation='softmax')(x)

model5 = Model(inputs=base_model.input, outputs=out)
model5.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model5.fit(rgb_train, y_train_cat, epochs=15, batch_size=64,
           validation_data=(rgb_test, y_test_cat))
_, acc5 = model5.evaluate(rgb_test, y_test_cat)
print(f"Transfer Learning Accuracy: {acc5:.4f}")

# Find two most confused classes from confusion matrix
from sklearn.metrics import confusion_matrix
y_pred5 = np.argmax(model5.predict(rgb_test), axis=1)
cm5 = confusion_matrix(y_test_enc, y_pred5)
np.fill_diagonal(cm5, 0)
flat_idx = np.argmax(cm5)
cls_true, cls_pred = divmod(flat_idx, 10)
print(f"Most confused pair: {CLASS_NAMES[cls_true]} → predicted as {CLASS_NAMES[cls_pred]}")


### 3.3 Multispectral Images

Apply your best model on multispectral images. You may use whichever image channels you wish, so long as you use more than just RGB (although you are not required to use any color channels). Calculate classification accuracy on the test data. Compare against results using RGB images.

How did adding multispectral channels impact your model's performance? Explain the role of additional spectral information in enhancing land cover classification.

In [ ]:
# Use all 13 bands (or pick a useful subset like B02,B03,B04,B08,B11,B12)
ms_train = images_train.astype(np.float32)
ms_test  = images_test.astype(np.float32)
ms_train = np.moveaxis(ms_train, 1, -1) / ms_train.max()  # (N,64,64,13)
ms_test  = np.moveaxis(ms_test,  1, -1) / ms_test.max()

# Reuse model5 architecture but change input shape
base_ms = EfficientNetB0(weights=None, include_top=False, input_shape=(64,64,13))
x = base_ms.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
out = Dense(10, activation='softmax')(x)

model_ms = Model(inputs=base_ms.input, outputs=out)
model_ms.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model_ms.fit(ms_train, y_train_cat, epochs=15, batch_size=64,
             validation_data=(ms_test, y_test_cat))
_, acc_ms = model_ms.evaluate(ms_test, y_test_cat)
print(f"Multispectral Accuracy: {acc_ms:.4f}")


## 4. Reflection Questions

What are your takeaways from tuning the parameters of the different models? What are your observations about increasing the number of training epochs? Did you run into any challenges or limitations when doing this? What was the impact of using dropout? How did the ensemble models compare to the other models? What kinds of challenges or limitations did you encounter when preparing and training the models for this assignment, and how might you address them in the future? How might you apply what you've learned about model tuning, dropout, and data processing to a different deep learning problem?